# Coastal flood step 01: intersections (set2)

Runs input-path fixes and vector-raster intersections for this set scenario.


In [ ]:
import os
from glob import glob
import geopandas
import pandas
import subprocess
from pathlib import Path
import sys
import numpy
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

# !{sys.executable} -m pip install "nismod-snail==0.5.3"

root = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))




In [ ]:
# processed_data_path = 'L:\Jamaica\Inputs'
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
output_path = base_path / "dphil_paper_3/results_coastal_set2"
processed_data_path = base_path / "dphil_paper_3/processed_data"
networks_path = base_path / "dphil_common_cross_cutting/common_incoming_data/networks/networks"
damage_curves_path = base_path / "dphil_common_cross_cutting/common_incoming_data/damage_curves"
robyn_libraries_path = base_path / "robyns_libraries"
data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"

jamaica_crs = 3448
jamaica_metric_grid_crs = "EPSG:3448"

mangroves_shapefile_candidates = [
    base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp",
    data_root / "landcover/mangroves_fn/mangroves.shp",
]
mangroves_shapefile = next((path for path in mangroves_shapefile_candidates if path.exists()), None)
if mangroves_shapefile is None:
    raise FileNotFoundError(
        "Could not find mangroves shapefile. Checked:\n" + "\n".join(str(path) for path in mangroves_shapefile_candidates)
    )

def resolve_network_asset_file(asset_relative_path):
    relative_asset_path = Path(asset_relative_path)
    asset_file_in_common_incoming_data = data_root / relative_asset_path
    asset_file_in_nested_networks_folder = data_root / "networks" / relative_asset_path

    if asset_file_in_common_incoming_data.exists():
        return asset_file_in_common_incoming_data
    if asset_file_in_nested_networks_folder.exists():
        return asset_file_in_nested_networks_folder

    raise FileNotFoundError(
        f"Could not find asset file '{relative_asset_path}'. Checked: \n"
        f"- {asset_file_in_common_incoming_data}\n"
        f"- {asset_file_in_nested_networks_folder}"
    )

# base_path = 'Z:\\jamaica\\Inputs'
# output_path = 'Z:\\jamaica\\Results'


In [ ]:
network_csv = data_root / "networks/network_layers_hazard_intersections_details.csv" 
hazard_csv = base_path / "dphil_paper_3/inputs/coastal_flood_rasters.csv"
damage_curves_csv = damage_curves_path / "asset_damage_curve_mapping.csv"
hazard_damage_parameters_csv = damage_curves_path / "hazard_damage_parameters.csv"
damage_results_folder = base_path / "dphil_paper_3/processed_data/direct_damages"


# network_csv = os.path.join(processed_data_path,
#                             "networks",
#                             "network_layers_hazard_intersections_details.csv")
# hazard_csv = os.path.join(processed_data_path,
#                             "coastal_flood_rasters.csv")
# damage_curves_csv = os.path.join(processed_data_path,
#                             "damage_curves",
# #                             "asset_damage_curve_mapping.csv")
# hazard_damage_parameters_csv = os.path.join(processed_data_path,
#                             "damage_curves",
#                             "hazard_damage_parameters.csv")
# damage_results_folder = "direct_damages"


In [ ]:

project_data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"
results_directory = Path(output_path)
results_directory.mkdir(parents=True, exist_ok=True)

network_layers_input_file = project_data_root / "networks/network_layers_hazard_intersections_details.csv"
network_layers_table = pandas.read_csv(network_layers_input_file)
network_layers_table = network_layers_table[["path"]].drop_duplicates().reset_index(drop=True)
network_layers_table["path"] = network_layers_table["path"].str.replace(
    r"^networks/", "networks/networks/", regex=True
)
network_layers_output_file = results_directory / "network_layers_fixed_for_intersections.csv"
network_layers_table.to_csv(network_layers_output_file, index=False)

coastal_rasters_input_file = base_path / "dphil_paper_3/inputs/coastal_flood_rasters.csv"
coastal_rasters_table = pandas.read_csv(coastal_rasters_input_file)
coastal_rasters_table["fname"] = coastal_rasters_table["path"]  # required by vector_raster_intersections.py
coastal_rasters_table["hazard"] = "coastal"  # align with hazard_damage_parameters.csv
hazard_layers_output_file = results_directory / "coastal_flood_rasters_fixed_for_intersections.csv"
coastal_rasters_table.to_csv(hazard_layers_output_file, index=False)

vector_details_csv = network_layers_output_file
raster_details_csv = hazard_layers_output_file
hazard_csv = hazard_layers_output_file

print("Network layers file:", vector_details_csv)
print("Hazard layers file:", raster_details_csv)
print("Summary hazard file:", hazard_csv)


In [ ]:
run_intersections = True  # Set to True is you want to run this process
if run_intersections is True:
    args = [
            "python",
            str(base_path / "robyns_libraries/vector_raster_intersections.py"),
            f"{vector_details_csv}",
            f"{raster_details_csv}",
            f"{output_path}"
            ]
    print ("* Start the processing of vector-raster intersections")
    print (args)
    subprocess.run(args)
print ("* Done with the processing of vector-raster intersections")
